In [1]:
pip install pandas numpy scikit-learn lightgbm xgboost shap mlflow


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 115.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 114.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.7/148.7 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
import pandas as pd, numpy as np

df = pd.read_csv("application_train.csv")
print(df.shape, "default rate:", df["TARGET"].mean())
print(df.isna().mean().sort_values(ascending=False).head(15))
print(df.dtypes.value_counts())

(307511, 122) default rate: 0.08072881945686496
COMMONAREA_AVG              0.698723
COMMONAREA_MODE             0.698723
COMMONAREA_MEDI             0.698723
NONLIVINGAPARTMENTS_MEDI    0.694330
NONLIVINGAPARTMENTS_MODE    0.694330
NONLIVINGAPARTMENTS_AVG     0.694330
FONDKAPREMONT_MODE          0.683862
LIVINGAPARTMENTS_AVG        0.683550
LIVINGAPARTMENTS_MEDI       0.683550
LIVINGAPARTMENTS_MODE       0.683550
FLOORSMIN_MODE              0.678486
FLOORSMIN_AVG               0.678486
FLOORSMIN_MEDI              0.678486
YEARS_BUILD_AVG             0.664978
YEARS_BUILD_MODE            0.664978
dtype: float64
float64    65
int64      41
object     16
Name: count, dtype: int64


In [5]:
# DAYS_EMPLOYED == 365243 is a placeholder for "not employed", not a real value
df["DAYS_EMPLOYED_ANOM"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)
df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)

from sklearn.model_selection import train_test_split
X = df.drop(columns=["TARGET", "SK_ID_CURR"])
y = df["TARGET"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
num_cols = X_train.select_dtypes("number").columns.tolist()
cat_cols = X_train.select_dtypes("object").columns.tolist()

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

def make_prep(scale):
    num_steps = [("imp", SimpleImputer(strategy="median"))]
    if scale:
        num_steps.append(("sc", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(num_steps), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore", min_frequency=100)),
        ]), cat_cols),
    ])

In [7]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from lightgbm import LGBMClassifier

# LightGBM handles NaN and categoricals natively
X_train_lgb = X_train.copy()
for c in cat_cols:
    X_train_lgb[c] = X_train_lgb[c].astype("category")

models = {
    "dummy": (DummyClassifier(strategy="prior"), X_train),
    "logreg": (Pipeline([("p", make_prep(True)),
               ("m", LogisticRegression(max_iter=1000, C=0.1))]), X_train),
    "tree": (Pipeline([("p", make_prep(False)),
             ("m", DecisionTreeClassifier(max_depth=6, min_samples_leaf=200))]), X_train),
    "random_forest": (Pipeline([("p", make_prep(False)),
                      ("m", RandomForestClassifier(n_estimators=200, max_depth=12,
                            min_samples_leaf=50, n_jobs=-1, random_state=42))]), X_train),
    "lightgbm": (LGBMClassifier(n_estimators=500, learning_rate=0.03, num_leaves=31,
                 subsample=0.8, colsample_bytree=0.7, random_state=42, verbose=-1), X_train_lgb),
}

cv = StratifiedKFold(5, shuffle=True, random_state=42)
rows = []
for name, (model, Xm) in models.items():
    r = cross_validate(model, Xm, y_train, cv=cv,
                       scoring=["roc_auc", "average_precision"], n_jobs=1)
    rows.append({
        "model": name,
        "roc_auc": r["test_roc_auc"].mean(),
        "pr_auc": r["test_average_precision"].mean(),
        "fit_sec": r["fit_time"].mean(),
    })
    print(rows[-1])

results = pd.DataFrame(rows).round(4)
results

{'model': 'dummy', 'roc_auc': np.float64(0.5), 'pr_auc': np.float64(0.08072908198907737), 'fit_sec': np.float64(0.11271028518676758)}
{'model': 'logreg', 'roc_auc': np.float64(0.7455784742730129), 'pr_auc': np.float64(0.22022160876485347), 'fit_sec': np.float64(13.523479509353638)}
{'model': 'tree', 'roc_auc': np.float64(0.712825002552159), 'pr_auc': np.float64(0.18552198410284895), 'fit_sec': np.float64(11.957419490814209)}
{'model': 'random_forest', 'roc_auc': np.float64(0.737425858189938), 'pr_auc': np.float64(0.2192195921297316), 'fit_sec': np.float64(83.66363563537598)}
{'model': 'lightgbm', 'roc_auc': np.float64(0.7568974110954315), 'pr_auc': np.float64(0.24019405639497085), 'fit_sec': np.float64(29.539339303970337)}


,model,roc_auc,pr_auc,fit_sec
0,dummy,0.5000,0.0807,0.1127
1,logreg,0.7456,0.2202,13.5235
2,tree,0.7128,0.1855,11.9574
3,random_forest,0.7374,0.2192,83.6636
4,lightgbm,0.7569,0.2402,29.5393
